<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Feature Detection and Object Tracking</b></h1>
</div>

## Requirements and Approach

This notebook mirrors the exact 13-task order of the executable notebook. Each section states the engineering requirement, implementation strategy, and acceptance condition for the corresponding stage of the tracking pipeline.

## Global Requirements

| Requirement | Implemented choice |
| --- | --- |
| Reference strategy | fixed first frame |
| Object initialization | manual bounding box |
| Local feature method | ORB |
| Binary descriptor distance | Hamming |
| Matching | BFMatcher with cross-check |
| Geometric mapping | $3\times3$ homography |
| Robust estimation | RANSAC |
| Minimum geometry support | 4 correspondences / 4 inliers |
| Invalid frame behavior | reject, do not fabricate a box |
| Sequence diagnostics | matches, inliers, inlier ratio |
| Memory policy | store compact tracking state; reload frames for figures |

## 1. Validate the Input Video and Output Paths

**Approach:** use repository-relative paths, fail explicitly if the video is missing, and create `outputs/figures/`.

**Acceptance:** input path exists as a file and the output directory is available.

## 2. Define the Initial Bounding Box and Tracking Parameters

**Approach:** create the four reference corners from $(row,col,height,width)=(24,46,170,160)$ and freeze ORB/RANSAC thresholds as named constants.

**Acceptance:** bounding-box array has shape $(4,1,2)$ and all parameters are explicit.

## 3. Initialize ORB and the Hamming-Distance Matcher

**Approach:** `cv2.ORB_create(nfeatures=1000)` plus `cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)`.

**Acceptance:** the detector and binary matcher are initialized once and reused.

## 4. Read and Validate the Reference Frame

**Approach:** open the video, read frame 0, convert BGR→grayscale, query frame count, and verify ROI bounds against image dimensions.

**Acceptance:** valid reference frame and ROI fully inside the image.

## 5. Detect Reference ORB Features Inside the Object Region

**Approach:** build an 8-bit binary ROI mask and call `detectAndCompute` only within the object box.

**Acceptance:** at least four keypoints, non-null descriptors, and ORB descriptor width 32 bytes.

## 6. Visualize the Reference Object and ORB Keypoints

**Approach:** draw the reference polygon and rich ORB keypoints, convert to RGB for Matplotlib, save the figure.

**Acceptance:** `reference_orb_keypoints.png` exists.

## 7. Define Frame Matching and RANSAC Homography Estimation

**Approach:** isolate all current-frame feature matching and robust geometry inside `match_and_estimate_homography()`. Return explicit failure reasons for missing descriptors, insufficient matches, failed homography, or insufficient inliers.

**Acceptance:** successful result contains finite $3\times3$ homography, inlier mask, match count, and inlier count.

## 8. Track the Object Throughout the Video

**Approach:** process frames sequentially, skip frame 0 because it is the reference, transform the four box corners with `cv2.perspectiveTransform`, and store only compact per-frame results.

**Acceptance:** each processed frame is represented exactly once as success or failure; successful boxes are finite.

## 9. Compute Tracking Summary Metrics

**Approach:** build NumPy arrays for frame indices, matches and inliers, then compute $r_i=n_{inlier,i}/n_{match,i}$ and global tracking success rate.

**Acceptance:** all ratios are finite and lie in $[0,1]$.

## 10. Visualize Representative Tracking Frames

**Approach:** use `read_video_frame()` to seek directly to selected successful frames rather than retaining every image in RAM.

**Acceptance:** representative montage saved as `representative_tracking_frames.png`.

## 11. Visualize RANSAC Inlier Matches

**Approach:** select the temporal middle successful frame, reload it, recompute feature matching, and pass the RANSAC mask to `cv2.drawMatches`.

**Acceptance:** `ransac_inlier_matches.png` contains only geometrically retained correspondences.

## 12. Analyze Matches, Inliers, and Inlier Ratio Across the Sequence

**Approach:** plot descriptor matches and RANSAC inliers together; separately plot inlier ratio versus frame index.

**Acceptance:** both sequence-level diagnostic files are generated.

## 13. Run Numerical and Output-file Validation Checks

**Approach:** enforce descriptor shape, frame-count consistency, finite $H$, finite transformed boxes, minimum matches/inliers, logical count ordering, ratio range, and expected output files.

**Acceptance:** any inconsistency raises an explicit exception; valid execution prints the final validation message.

## Requirement-to-Code Traceability

| Task | Main implementation location |
| ---: | --- |
| 1 | video/output-path validation cell |
| 2 | ROI + tracking-constant definition cell |
| 3 | ORB + BFMatcher initialization cell |
| 4 | reference-frame loading/ROI-bound validation |
| 5 | ROI mask + reference `detectAndCompute` |
| 6 | reference-keypoint figure cell |
| 7 | `match_and_estimate_homography()` |
| 8 | full video tracking loop |
| 9 | sequence summary arrays and metrics |
| 10 | `read_video_frame()` + representative montage |
| 11 | representative RANSAC-match visualization |
| 12 | temporal match/inlier/ratio plots |
| 13 | final numerical and output validation |

Every numbered task in the Implementation notebook is immediately followed by executable code.

## Scope Boundaries

The method assumes that the visible object region can be related to the reference by a planar projective transformation or a sufficiently close approximation. It does not use optical flow, a learned detector, a dedicated tracker, temporal motion prediction, descriptor ratio tests, bundle adjustment, or non-rigid deformation models.